In [1]:
import os

# Create the folder structure
os.makedirs('src', exist_ok=True)
os.makedirs('tests', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Create the empty __init__.py files (CRITICAL for Python imports)
with open('src/__init__.py', 'w') as f: pass
with open('tests/__init__.py', 'w') as f: pass

print("✅ Folders and __init__ files created!")

✅ Folders and __init__ files created!


The Engine (src/stat_engine.py)

In [2]:
%%writefile src/stat_engine.py
import math
from typing import List, Union, Any, Dict

class StatEngine:
    def __init__(self, raw_data: List[Any]):
        self.data = []
        for item in raw_data:
            try:
                if item is not None:
                    self.data.append(float(item))
            except (ValueError, TypeError):
                continue
        if not self.data:
            raise ValueError("ZeroDivisionError Prevention: No valid numeric data found.")

    def get_mean(self) -> float:
        return sum(self.data) / len(self.data)

    def get_median(self) -> float:
        s = sorted(self.data)
        n = len(s)
        mid = n // 2
        return (s[mid-1] + s[mid]) / 2.0 if n % 2 == 0 else float(s[mid])

    def get_mode(self) -> Union[List[float], str]:
        counts = {}
        for x in self.data:
            counts[x] = counts.get(x, 0) + 1
        max_f = max(counts.values())
        if max_f == 1 and len(self.data) > 1:
            return "Specific Message: All values are unique; no mode exists."
        return sorted([k for k, v in counts.items() if v == max_f])

    def get_variance(self, is_sample: bool = True) -> float:
        n = len(self.data)
        if is_sample and n <= 1:
            raise ZeroDivisionError("Bessel's Correction requires n > 1.")
        mu = self.get_mean()
        sq_diff = sum((x - mu)**2 for x in self.data)
        return sq_diff / (n - 1 if is_sample else n)

    def get_standard_deviation(self, is_sample: bool = True) -> float:
        return math.sqrt(self.get_variance(is_sample))

    def get_outliers(self, threshold: float = 2.0) -> List[float]:
        mu, sigma = self.get_mean(), self.get_standard_deviation(is_sample=True)
        return [x for x in self.data if abs(x - mu) > (threshold * sigma)]

Writing src/stat_engine.py


The Simulation (src/monte_carlo.py)

In [8]:
%%writefile src/monte_carlo.py
import random

def simulate_crashes(days: int) -> float:
    PROBABILITY = 0.045
    crashes = sum(1 for _ in range(days) if random.random() < PROBABILITY)
    return crashes / days

def run_simulation_report():
    print("\n--- Monte Carlo Simulation: Server Crashes (Target: 4.5%) ---")
    for days in [30, 1000, 10000]:
        prob = simulate_crashes(days)
        print(f"Days: {days:6} | Simulated Prob: {prob:.4%} | Error: {abs(0.045 - prob):.4%}")

Overwriting src/monte_carlo.py


The Unit Tests (tests/test_stat_engine.py)

In [4]:
%%writefile tests/test_stat_engine.py
import unittest
from src.stat_engine import StatEngine

class TestStatEngine(unittest.TestCase):
    def test_bessel(self):
        engine = StatEngine([10, 20])
        self.assertEqual(engine.get_variance(is_sample=True), 50.0)

    def test_median_even(self):
        engine = StatEngine([1, 2, 3, 4])
        self.assertEqual(engine.get_median(), 2.5)

if __name__ == "__main__":
    unittest.main()

Writing tests/test_stat_engine.py


The Data (data/sample_salaries.json)

In [5]:
%%writefile data/sample_salaries.json
[55000, 48000, 52000, 61000, 59000, 50000, 45000, 300000, 1000000]

Writing data/sample_salaries.json


The Main Entry (main.py)

In [6]:
%%writefile main.py
import json
from src.stat_engine import StatEngine
from src.monte_carlo import run_simulation_report

def main():
    with open('data/sample_salaries.json', 'r') as f:
        data = json.load(f)
    engine = StatEngine(data)
    print(f"Mean Salary: ${engine.get_mean():,.2f}")
    print(f"Outliers: {engine.get_outliers(threshold=2)}")
    run_simulation_report()

if __name__ == "__main__":
    main()

Writing main.py


Run the Engine

In [7]:
# 1. Run the Tests (The Verification)
!python -m unittest discover tests

# 2. Run the Main Analysis (The Results)
!python main.py

..
----------------------------------------------------------------------
Ran 2 tests in 0.000s

OK
Mean Salary: $185,555.56
Outliers: [1000000.0]

--- Monte Carlo Simulation: Server Crashes (Target: 4.5%) ---
Days:     30 | Simulated Prob: 3.3333% | Error: 1.1667%
Days:   1000 | Simulated Prob: 4.4000% | Error: 0.1000%
Days:  10000 | Simulated Prob: 4.2900% | Error: 0.2100%
